### Installation des dépendances

Configuration adaptée du notebook `LLM setup` fourni dans le modèle de projet pour exécuter [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) et le pipeline RAG sur Google Colab.

- L'environnement Colab est configuré avec un GPU T4 et un runtime Python 3.11.
- La cellule installe les dépendances nécessaires avec `uv`, en fixant les versions de certaines bibliothèques et en ajoutant celles nécessaires au RAG.
- L'installation est ignorée lorsque PandasAI 3.0.0 et les trois bibliothèques RAG sont déjà présents. Après une première installation, un redémarrage de la session Colab est nécessaire.
- Hors Colab, l'installation est ignorée et l'environnement peut être configuré localement avec `uv sync --extra llm --extra rag`.

In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IN_COLAB = "google.colab" in sys.modules


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


# Skipped once the packages are installed, so this cell can be re-run after restarting the session.
if IN_COLAB and installed_version("pandasai") != "3.0.0":
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")

    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])

    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==2.3.3",
        "transformers==5.17.0",
        "accelerate==1.15.0",
        "bitsandbytes==0.50.2",
        "peft==0.21.2",
        "sentence-transformers>=6.1.0",
        "requests",
        "pandasai==3.0.0",

        # Dépendances supplémentaires pour le RAG
        "faiss-cpu>=1.15.1",
        "langchain-text-splitters>=1.1.3",
        "datasets>=4,<6",
    ]

    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")

### Récupération du dépôt GitHub

Cette cellule clone le dépôt GitHub de notre équipe dans `/content/repo` afin de rendre le package `ift6758` accessible depuis Colab.

- Le clonage est effectué uniquement si le dossier cible n'existe pas déjà.
- Le dossier du dépôt est ajouté à `sys.path` pour permettre l'importation des modules Python du projet.
- Hors Colab, cette étape est ignorée.

In [2]:
import os

if IN_COLAB:
    from google.colab import userdata

    REPO = "ammou35/LNH-data-science"
    REPO_DIR = "/content/repo"

    if not os.path.exists(REPO_DIR):
        subprocess.check_call(["git", "clone", f"https://github.com/{REPO}", REPO_DIR])

    PROJECT_DIR = os.path.join(REPO_DIR, "data_science_project")
    sys.path.insert(0, PROJECT_DIR)

### Vérification de l'environnement

- Vérification de la version de Python et des principales dépendances.
- Affichage des versions utilisées pour documenter l'environnement et faciliter la reproductibilité des résultats.

In [3]:
import sys
import torch
import numpy as np
import pandas as pd
from importlib.metadata import version

if sys.version_info[:2] != (3, 11):
    raise RuntimeError("A Python 3.11 runtime is required.")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Transformers:", version("transformers"))
print("Accelerate:", version("accelerate"))
print("PandasAI:", version("pandasai"))

Python: 3.11.16
PyTorch: 2.14.0
NumPy: 1.26.4
pandas: 2.3.3
Transformers: 5.17.0
Accelerate: 1.15.0
PandasAI: 3.0.0


### Vérification du GPU

- Si un GPU est disponible, son nom est affiché.
- Sinon, un message rappelle de sélectionner un runtime GPU T4 dans les paramètres de Google Colab.

In [4]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).")

No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).


### Chargement du modèle Qwen

- Le modèle Qwen3-4B-Instruct-2507 est chargé à l'aide de la fonction `load_model()` du package `ift6758`.
- Lors de la première exécution, environ 8 Go de poids sont téléchargés dans le cache Hugging Face.
- Le périphérique d'exécution et le type de données numériques du modèle sont affichés après le chargement.
- Une première requête sur le hockey permet de vérifier la génération de texte avec le gabarit de conversation de Qwen.

In [5]:
from ift6758.llm import load_model

model, tokenizer = load_model()
print(f"Loaded on {model.device} ({model.dtype})")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded on mps:0 (torch.float16)


In [6]:
def ask_qwen(prompt, system="You are a helpful assistant.", max_new_tokens=512):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask_qwen("In one sentence, what is a hat trick in hockey?"))

In hockey, a hat trick occurs when a player scores one goal, records one assist, and records one blocked shot in the same game.


### Chargement des données brutes de la NHL

Cette cellule récupère les données brutes de la NHL de 2016 à 2023 avec le code d'acquisition implémenté à la tâche 1.

In [7]:
from ift6758.data.nhl import download_seasons

download_seasons(first=2016, last=2023)

2016 regular: 1230game [00:04, 250.35game/s] 
2016 playoffs: 87game [00:08,  9.86game/s]
2017 regular: 1271game [00:04, 268.62game/s] 
2017 playoffs: 84game [00:08,  9.66game/s]
2018 regular: 1271game [00:04, 276.77game/s] 
2018 playoffs: 87game [00:06, 13.21game/s]
2019 regular: 1082game [00:04, 243.50game/s]
2019 playoffs: 86game [00:08, 10.02game/s]
2020 regular: 868game [00:04, 207.73game/s] 
2020 playoffs: 84game [00:08, 10.04game/s]
2021 regular: 1312game [00:04, 279.19game/s] 
2021 playoffs: 89game [00:06, 13.77game/s]
2022 regular: 1312game [00:04, 285.66game/s] 
2022 playoffs: 88game [00:08, 10.84game/s]
2023 regular: 1312game [00:04, 271.72game/s] 
2023 playoffs: 88game [00:07, 11.16game/s]


### Préparation du match de référence

Cette cellule récupère les données réelles d'un match NHL connu afin de disposer d'une référence indépendante pour la comparaison.

- Le match `2019020001`, appartenant à la saison `2019-2020`, est sélectionné comme référence.
- Si son fichier JSON existe déjà, les données sont chargées localement. Sinon, elles sont téléchargées depuis l'API NHL et sauvegardées.
- L'identifiant du match et la présence d'une liste d'événements sont vérifiés.
- Le dictionnaire game contient les données de référence utilisées dans les tests suivants.

In [8]:
import json
import ift6758
from pathlib import Path
import requests

SEASON_ID = 2019
GAME_ID = 2019020001

PACKAGE_DIR = Path(ift6758.__file__).resolve().parent
DATA_DIR = PACKAGE_DIR / "data"

raw_path = DATA_DIR / "raw" / f"{GAME_ID}.json"

if not raw_path.exists():
    response = requests.get(
        f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play",
        timeout=30,
    )
    response.raise_for_status()

    downloaded = response.json()
    assert downloaded["id"] == GAME_ID
    assert isinstance(downloaded["plays"], list)

    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(json.dumps(downloaded), encoding="utf-8")

game = json.loads(raw_path.read_text(encoding="utf-8"))

assert game["id"] == GAME_ID
assert isinstance(game["plays"], list)

print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

OTT at TOR 2019-10-02


### Génération de la logique de transformation de caractéristiques avec Qwen

In [9]:
import ift6758
from pathlib import Path
from IPython.display import Markdown, display

prompt = f"""Write a Python function compute_distance_llm(df).

Requirements from our project:
- Use pandas and numpy. Include imports and a docstring.
- df is a pandas dataframe containing NHL shot and goal events.
- The dataframe includes ice coordinates in columns x and y.
- Compute the distance between each shot and the net being targeted.
- Explain any assumptions about net coordinates and shooting direction
  in the docstring.
- Preserve missing values instead of replacing them with 0 or False.
- Return the original dataframe augmented with a "distance" column.
Return only the function and imports in a Python code block.
"""

generated_code = ask_qwen(prompt, system="You write concise, correct Python code.", max_new_tokens=2000)

PACKAGE_DIR = Path(ift6758.__file__).resolve().parent
DATA_DIR = PACKAGE_DIR / "data"

display(Markdown(generated_code))

generated_code_path = PACKAGE_DIR / "llm" / "compute_distance_llm.py"
generated_code_path.parent.mkdir(parents=True, exist_ok=True)
generated_code = generated_code.strip().removeprefix("```python").removesuffix("```").strip()
generated_code_path.write_text(generated_code + "\n", encoding="utf-8")

```python
import pandas as pd
import numpy as np

def compute_distance_llm(df):
    """
    Compute the distance between each shot and the net being targeted.
    
    Assumes the net is centered at (0, 0) with a standard NHL net size of 6 feet (1.83 meters) in width
    and 4 feet (1.22 meters) in height. The net is centered at the origin, and the shooting direction
    is assumed to be along the x-axis (from right to left), with positive x values indicating shots
    from the right side of the ice, and negative x values from the left. The y-coordinate represents
    vertical distance from the center of the net.
    
    The distance is computed as the Euclidean distance from the shot point (x, y) to the net center (0, 0).
    This assumes the net is centered at the origin and that the shot is aimed at the center of the net.
    
    Missing values (NaN) in x or y are preserved without replacement.
    
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing NHL shot and goal events with columns 'x' and 'y' representing ice coordinates.
    
    Returns
    -------
    pandas.DataFrame
        The original DataFrame augmented with a 'distance' column representing the Euclidean distance
        from each shot to the net center (0, 0).
    """
    # Compute Euclidean distance from (x, y) to (0, 0)
    df['distance'] = np.sqrt(df['x']**2 + df['y']**2)
    return df
```

1405

### Intégration de la fonction compute_distance générée par Qwen

Cette cellule permet l'intégration du code proposé pour le calcul des distances au filet par Qwen pour son évaluation.

In [19]:
from ift6758.visualizations.simple_viz import compute_distance

# Remplacer la ligne suivante par le code généré par Qwen dans la sortie de la
# cellule précédente (en faisant attention à ne pas copier la dernière ligne) :
import pandas as pd
import numpy as np

def compute_distance_llm(df):
    """
    Compute the Euclidean distance between each shot and the net center.

    Assumes the net is centered at (0, 0) on the ice, with the goal located at the origin.
    The shot coordinates (x, y) are relative to the center of the rink, with positive x
    indicating the direction of the shot (towards the net), and y indicating vertical position.
    The net is assumed to be centered at (0, 0), and the distance is calculated as the
    Euclidean distance from the shot point to the net center.

    Missing values in x or y are preserved (not replaced with 0 or False).

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing NHL shot and goal events with columns 'x' and 'y' representing
        ice coordinates of the shot.

    Returns
    -------
    pandas.DataFrame
        The original DataFrame augmented with a 'distance' column representing the
        Euclidean distance from each shot to the net center (0, 0).
    """
    # Net is assumed to be centered at (0, 0)
    df['distance'] = np.sqrt(df['x']**2 + df['y']**2)
    return df

### Comparaison de la transformation des caractéristiques

Cette cellule compare les distances calculées par notre implémentation
manuelle et celle générée par Qwen à partir des mêmes événements.

In [20]:
from ift6758.data.tidy_data import game_to_pandas
functions = {
    "Manual": compute_distance,
    "Qwen": compute_distance_llm,
}

shots = game_to_pandas(game)

results = {}
dataframes = {}
errors = {}

for implementation, function in functions.items():
    try:
        transformed = function(shots.copy())
        dataframes[implementation] = transformed

        results[implementation] = {
            "missing": transformed["distance"].isna().sum(),
        }

        display(transformed[["x", "y", "distance"]].head(10))

    except Exception as error:
        errors[implementation] = error
        results[implementation] = {"missing": None}

,player_name,goalie_name
0,Brady Tkachuk,Frederik Andersen
1,Morgan Rielly,Craig Anderson
2,Dylan DeMelo,Frederik Andersen
3,Morgan Rielly,Craig Anderson
4,Tyson Barrie,Craig Anderson
5,Tyson Barrie,Craig Anderson
6,Cody Ceci,Craig Anderson
7,Andreas Johnsson,Craig Anderson
8,Mitch Marner,Craig Anderson
9,William Nylander,Craig Anderson


,x,y,distance
0,85,-1,4
1,-32,-2,57
2,63,-6,26
3,-59,-20,36
4,-42,-29,55
5,-52,-7,37
6,-38,38,63
7,-76,-14,19
8,-63,-31,40
9,-78,13,17


,x,y,distance
0,85,-1,85.005882
1,-32,-2,32.062439
2,63,-6,63.285069
3,-59,-20,62.297673
4,-42,-29,51.039201
5,-52,-7,52.469038
6,-38,38,53.740115
7,-76,-14,77.278716
8,-63,-31,70.213959
9,-78,13,79.075913


### Résultats de la comparaison

Cette cellule regroupe les résultats des vérifications effectuées sur les deux implémentations dans un tableau récapitulatif.

Cette cellule présente le nombre de distances manquantes pour chaque implémentation et vérifie si les distances calculées à partir des mêmes événements sont équivalentes.

In [21]:
comparison = pd.DataFrame.from_dict(results, orient="index")
comparison.index.name = "Implémentation"

display(comparison)

if all(name in dataframes for name in functions):
    print("Distances identiques :", np.allclose(dataframes["Manual"]["distance"], dataframes["Qwen"]["distance"], equal_nan=True))

,missing
Implémentation,
Manual,0
Qwen,0


Distances identiques : False


### Génération du code de nettoyage des données avec Qwen

Cette cellule demande à Qwen3-4B-Instruct-2507 de générer une fonction de nettoyage des événements play-by-play d'un match NHL, afin de comparer son implémentation à notre fonction manuelle `game_to_pandas`.

In [22]:
import ift6758
from pathlib import Path
from IPython.display import Markdown, display

prompt = f"""Write a Python function game_to_pandas_llm(game).

Requirements from our project:
- Use pandas. Include imports and a docstring.
- game is a full NHL play-by-play JSON dictionary.
- Convert the game events into a pandas dataframe.
- Include only "shot-on-goal" and "goal" events.
- Use one row per event, uniquely identified by the "game_id" and "event_id" columns.
- Include "season" as the starting year of the season (e.g., 2017 for 2017-2018).
- Include "game_type" using the NHL gameType field (2 for regular season, 3 for playoffs).
- Include period, time in period, shooting team, event type,
  ice coordinates, shooter name, goalie name and shot type.
- Use rosterSpots to retrieve player names from player IDs.
- Include whether the shot was taken at an empty net.
- Use situationCode to derive the actual manpower strength
  and whether the shooting team is at even strength,
  on a power play or shorthanded.
- situationCode follows this format:
  [away goalie][away skaters][home skaters][home goalie].
- Preserve missing values instead of replacing them with 0 or False.
- Return the formatted pandas dataframe.
Return only the function and imports in a Python code block.
"""

generated_code = ask_qwen(prompt, system="You write concise, correct Python code.", max_new_tokens=2000)

PACKAGE_DIR = Path(ift6758.__file__).resolve().parent
DATA_DIR = PACKAGE_DIR / "data"

display(Markdown(generated_code))

generated_code_path = PACKAGE_DIR / "llm" / "game_to_pandas_llm.py"
generated_code_path.parent.mkdir(parents=True, exist_ok=True)
generated_code = generated_code.strip().removeprefix("```python").removesuffix("```").strip()
generated_code_path.write_text(generated_code + "\n", encoding="utf-8")

```python
import pandas as pd
import json

def game_to_pandas_llm(game):
    """
    Convert NHL play-by-play JSON data into a pandas DataFrame containing only shot-on-goal and goal events.
    
    Parameters:
    game (dict): A full NHL play-by-play JSON dictionary.
    
    Returns:
    pd.DataFrame: A DataFrame with one row per shot-on-goal or goal event, including:
        - game_id, event_id (unique identifiers)
        - season (starting year of the season, e.g., 2017 for 2017-2018)
        - game_type (2 for regular season, 3 for playoffs)
        - period, time_in_period
        - shooting_team, event_type
        - ice_coordinates (x, y)
        - shooter_name, goalie_name
        - shot_type
        - empty_net (boolean)
        - manpower_strength (e.g., 'even', 'power play', 'shorthanded')
        - at_even_strength (boolean)
    
    Missing values are preserved as NaN.
    """
    events = []
    
    # Extract season and game_type
    season = game['gameData']['gameData']['season'].split('-')[0]
    game_type = game['gameData']['gameData']['gameType']
    
    # Helper to get player name from rosterSpot
    def get_player_name(player_id, rosters):
        for roster in rosters:
            for player in roster['players']:
                if player['playerId'] == player_id:
                    return player['name']['first'] + ' ' + player['name']['last']
        return None
    
    # Process each event
    for event in game['gameData']['plays']:
        event_type = event['event']['type']
        if event_type not in ['shot-on-goal', 'goal']:
            continue
            
        # Extract relevant fields
        game_id = event['gameId']
        event_id = event['event']['id']
        period = event['event']['period']
        time_in_period = event['event']['time']
        
        # Determine shooting team (home or away)
        team = event['event']['team']
        shooting_team = 'home' if team == 'home' else 'away'
        
        # Extract ice coordinates
        x = event['event']['x'] if 'x' in event['event'] else None
        y = event['event']['y'] if 'y' in event['event'] else None
        ice_coordinates = (x, y)
        
        # Extract shooter and goalie
        shooter_id = event['event']['shooterId'] if 'shooterId' in event['event'] else None
        goalie_id = event['event']['goalieId'] if 'goalieId' in event['event'] else None
        
        shooter_name = None
        goalie_name = None
        
        # Get shooter and goalie names from rosters
        if 'rosters' in game and game['rosters']:
            shooter_name = get_player_name(shooter_id, game['rosters'])
            goalie_name = get_player_name(goalie_id, game['rosters'])
        
        # Extract shot type
        shot_type = event['event']['shotType'] if 'shotType' in event['event'] else None
        
        # Determine empty net
        empty_net = False
        situation_code = event['event']['situationCode'] if 'situationCode' in event['event'] else None
        if situation_code:
            # Format: [away goalie][away skaters][home skaters][home goalie]
            # If last character is '0', it's empty net
            if situation_code[-1] == '0':
                empty_net = True
        
        # Derive manpower strength and even strength status
        manpower_strength = None
        at_even_strength = False
        
        if situation_code:
            # Parse situationCode: [away goalie][away skaters][home skaters][home goalie]
            # Each group is 1-3 characters; 0 = none, 1-3 = players
            parts = situation_code
            away_goalie = parts[0] if len(parts) > 0 else None
            away_skaters = parts[1] if len(parts) > 1 else None
            home_skaters = parts[2] if len(parts) > 2 else None
            home_goalie = parts[3] if len(parts) > 3 else None
            
            # Count skaters
            total_away_skaters = int(away_skaters) if away_skaters else 0
            total_home_skaters = int(home_skaters) if home_skaters else 0
            
            # Determine manpower
            if total_away_skaters == 0 and total_home_skaters == 0:
                manpower_strength = 'even'
                at_even_strength = True
            elif total_away_skaters > 0:
                manpower_strength = 'power play'
                at_even_strength = False
            elif total_home_skaters > 0:
                manpower_strength = 'shorthanded'
                at_even_strength = False
        
        # Add to events list
        events.append({
            'game_id': game_id,
            'event_id': event_id,
            'season': season,
            'game_type': game_type,
            'period': period,
            'time_in_period': time_in_period,
            'shooting_team': shooting_team,
            'event_type': event_type,
            'ice_coordinates': ice_coordinates,
            'shooter_name': shooter_name,
            'goalie_name': goalie_name,
            'shot_type': shot_type,
            'empty_net': empty_net,
            'manpower_strength': manpower_strength,
            'at_even_strength': at_even_strength
        })
    
    # Convert to DataFrame
    df = pd.DataFrame(events)
    
    return df
```

5261

### Intégration de la fonction générée par Qwen

Cette cellule permet l'intégration du code proposé pour le nettoyage des donnés par Qwen pour son évaluation.

In [23]:
# Remplacer la ligne suivante par le code généré par Qwen dans la sortie de la
# cellule précédente (en faisant attention à ne pas copier la dernière ligne) :
import pandas as pd
import json

def game_to_pandas_llm(game):
    """
    Convert NHL play-by-play JSON data into a pandas DataFrame containing only shot-on-goal and goal events.

    Parameters:
    game (dict): A full NHL play-by-play JSON dictionary.

    Returns:
    pd.DataFrame: A DataFrame with one row per shot-on-goal or goal event, including:
        - game_id, event_id (unique identifiers)
        - season (starting year of the season, e.g., 2017 for 2017-2018)
        - game_type (2 for regular season, 3 for playoffs)
        - period, time_in_period
        - shooting_team, event_type
        - ice_coordinates (x, y)
        - shooter_name, goalie_name
        - shot_type
        - empty_net (boolean)
        - manpower_strength (e.g., 'even', 'power play', 'shorthanded')
        - at_even_strength (boolean)

    Missing values are preserved as NaN.
    """
    events = []

    # Extract season and game_type
    season = game['gameData']['gameData']['season'].split('-')[0]
    game_type = game['gameData']['gameData']['gameType']

    # Helper to get player name from rosterSpot
    def get_player_name(player_id, rosters):
        for roster in rosters:
            for player in roster['players']:
                if player['playerId'] == player_id:
                    return player['name']['first'] + ' ' + player['name']['last']
        return None

    # Process each event
    for event in game['gameData']['plays']:
        event_type = event['event']['type']
        if event_type not in ['shot-on-goal', 'goal']:
            continue

        # Extract relevant fields
        game_id = event['gameId']
        event_id = event['event']['id']
        period = event['event']['period']
        time_in_period = event['event']['time']

        # Determine shooting team (home or away)
        team = event['event']['team']
        shooting_team = 'home' if team == 'home' else 'away'

        # Extract ice coordinates
        x = event['event']['x'] if 'x' in event['event'] else None
        y = event['event']['y'] if 'y' in event['event'] else None
        ice_coordinates = (x, y)

        # Extract shooter and goalie
        shooter_id = event['event']['shooterId'] if 'shooterId' in event['event'] else None
        goalie_id = event['event']['goalieId'] if 'goalieId' in event['event'] else None

        shooter_name = None
        goalie_name = None

        # Get shooter and goalie names from rosters
        if 'rosters' in game and game['rosters']:
            shooter_name = get_player_name(shooter_id, game['rosters'])
            goalie_name = get_player_name(goalie_id, game['rosters'])

        # Extract shot type
        shot_type = event['event']['shotType'] if 'shotType' in event['event'] else None

        # Determine empty net
        empty_net = False
        situation_code = event['event']['situationCode'] if 'situationCode' in event['event'] else None
        if situation_code:
            # Format: [away goalie][away skaters][home skaters][home goalie]
            # If last character is '0', it's empty net
            if situation_code[-1] == '0':
                empty_net = True

        # Derive manpower strength and even strength status
        manpower_strength = None
        at_even_strength = False

        if situation_code:
            # Parse situationCode: [away goalie][away skaters][home skaters][home goalie]
            # Each group is 1-3 characters; 0 = none, 1-3 = players
            parts = situation_code
            away_goalie = parts[0] if len(parts) > 0 else None
            away_skaters = parts[1] if len(parts) > 1 else None
            home_skaters = parts[2] if len(parts) > 2 else None
            home_goalie = parts[3] if len(parts) > 3 else None

            # Count skaters
            total_away_skaters = int(away_skaters) if away_skaters else 0
            total_home_skaters = int(home_skaters) if home_skaters else 0

            # Determine manpower
            if total_away_skaters == 0 and total_home_skaters == 0:
                manpower_strength = 'even'
                at_even_strength = True
            elif total_away_skaters > 0:
                manpower_strength = 'power play'
                at_even_strength = False
            elif total_home_skaters > 0:
                manpower_strength = 'shorthanded'
                at_even_strength = False

        # Add to events list
        events.append({
            'game_id': game_id,
            'event_id': event_id,
            'season': season,
            'game_type': game_type,
            'period': period,
            'time_in_period': time_in_period,
            'shooting_team': shooting_team,
            'event_type': event_type,
            'ice_coordinates': ice_coordinates,
            'shooter_name': shooter_name,
            'goalie_name': goalie_name,
            'shot_type': shot_type,
            'empty_net': empty_net,
            'manpower_strength': manpower_strength,
            'at_even_strength': at_even_strength
        })

    # Convert to DataFrame
    df = pd.DataFrame(events)

    return df


### Préparation des résultats de comparaison

Cette cellule initialise les structures utilisées pour conserver les résultats des tests.

In [24]:
functions = {
    "Manual": game_to_pandas,
    "Qwen": game_to_pandas_llm,
}

plays = game["plays"]
events = pd.json_normalize(plays)
keep = events["typeDescKey"].isin(["shot-on-goal", "goal"])

assert len(events) == len(plays), "Flattening changed the number of events."

results = {}
dataframes = {}
errors = {}

### Comparaison des implémentations

Cette cellule présente les résultats des vérifications effectuées sur les deux implémentations, ainsi que les inspections des valeurs manquantes et le résumé des tirs et des buts par équipe.

In [30]:
for implementation, function in functions.items():
    results[implementation] = {
        "events": None,
        "identifiers": None,
        "missing": None,
        "summary": None,
    }

    try:
        shots = function(game)
        dataframes[implementation] = shots

        # Vérification des événements conservés
        results[implementation]["events"] = False
        assert len(shots) == int(keep.sum()), "Cleaning changed the number of retained events."
        assert set(shots["event_id"]) == set(events.loc[keep, "eventId"]), "Incorrect retained events."
        results[implementation]["events"] = True

        # Vérification des identifiants
        results[implementation]["identifiers"] = False
        assert shots[["game_id", "event_id"]].notna().all().all(), "Missing event identifier."
        assert not shots.duplicated(["game_id", "event_id"]).any(), "Duplicate event identifiers."
        results[implementation]["identifiers"] = True

        # Vérification des valeurs manquantes
        results[implementation]["missing"] = False
        display(shots.isna().sum().rename("missing_count").to_frame())
        shots_with_coordinates = shots.dropna(subset=["x", "y"])
        print("Rows excluded from a coordinate-based analysis:", len(shots) - len(shots_with_coordinates))
        print("Rows kept in the full shots table:", len(shots))
        results[implementation]["missing"] = True

        # Résumé des tirs et des buts par équipe
        results[implementation]["summary"] = False
        summary = shots.groupby("team", dropna=False).agg(shot_events=("event_id", "size"), goals=("is_goal", "sum"))
        display(summary)
        results[implementation]["summary"] = True

        # Inspection du DataFrame
        display(shots.head(10))

    except Exception as error:
        errors[implementation] = error
        if all(value is None for value in results[implementation].values()):
            results[implementation]["events"] = "Erreur d'exécution"

,missing_count
season,0
game_id,0
event_id,0
game_type,0
period,0
period_type,0
time_in_period,0
team,0
team_id,0
event_type,0


Rows excluded from a coordinate-based analysis: 0
Rows kept in the full shots table: 68


,shot_events,goals
team,,
OTT,26,3
TOR,42,5


,season,game_id,event_id,game_type,period,period_type,time_in_period,team,team_id,event_type,is_goal,x,y,player_name,goalie_name,shot_type,zone,empty_net,strength
0,2019,2019020001,10,2,1,REG,00:25,OTT,9,goal,True,85,-1,Brady Tkachuk,Frederik Andersen,tip-in,O,False,0
1,2019,2019020001,15,2,1,REG,01:31,TOR,10,shot-on-goal,False,-32,-2,Morgan Rielly,Craig Anderson,snap,O,False,1
2,2019,2019020001,20,2,1,REG,03:23,OTT,9,shot-on-goal,False,63,-6,Dylan DeMelo,Frederik Andersen,snap,O,False,0
3,2019,2019020001,21,2,1,REG,03:56,TOR,10,shot-on-goal,False,-59,-20,Morgan Rielly,Craig Anderson,wrist,O,False,0
4,2019,2019020001,24,2,1,REG,04:47,TOR,10,shot-on-goal,False,-42,-29,Tyson Barrie,Craig Anderson,slap,O,False,0
5,2019,2019020001,25,2,1,REG,04:53,TOR,10,shot-on-goal,False,-52,-7,Tyson Barrie,Craig Anderson,slap,O,False,0
6,2019,2019020001,28,2,1,REG,05:31,TOR,10,shot-on-goal,False,-38,38,Cody Ceci,Craig Anderson,wrist,O,False,0
7,2019,2019020001,29,2,1,REG,05:52,TOR,10,shot-on-goal,False,-76,-14,Andreas Johnsson,Craig Anderson,wrist,O,False,0
8,2019,2019020001,31,2,1,REG,06:02,TOR,10,shot-on-goal,False,-63,-31,Mitch Marner,Craig Anderson,slap,O,False,1
9,2019,2019020001,37,2,1,REG,07:11,TOR,10,shot-on-goal,False,-78,13,William Nylander,Craig Anderson,tip-in,O,False,1


### Inspection des résultats

Cette cellule affiche les dimensions des DataFrames obtenus et les éventuelles erreurs rencontrées lors de l'exécution des deux implémentations.

In [31]:
for implementation, shots in dataframes.items():
    print(f"{implementation}:", shots.shape)

for implementation, error in errors.items():
    print(f"{implementation}: {type(error).__name__} - {error}")

Manual: (68, 19)
Qwen: KeyError - 'gameData'


### Bilan des vérifications

Cette cellule rassemble les résultats des tests dans un tableau comparatif afin d'identifier les vérifications réussies, échouées ou non exécutées pour chaque implémentation.

In [32]:
comparison = pd.DataFrame.from_dict(results, orient="index")

comparison = comparison.rename(columns={
    "events": "Événements",
    "identifiers": "Identifiants",
    "missing": "Valeurs manquantes",
    "summary": "Résumé par équipe",
})

comparison = comparison.replace({True: "Succès", False: "Échec", None: "Non testé"})
comparison.index.name = "Implémentation"

display(comparison)

,Événements,Identifiants,Valeurs manquantes,Résumé par équipe
Implémentation,,,,
Manual,Succès,Succès,Succès,Succès
Qwen,Erreur d'exécution,Non testé,Non testé,Non testé
